In [1]:
# Cell 1 -- check what GPU you got
!nvidia-smi

Tue Aug 25 11:19:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Cell 2 -- clone and install
!git clone https://github.com/swapnilanilkale/panorama-onco.git
%cd panorama-onco
# Colab runs Python 3.13; ADR-0004 pins <3.13 for local ecosystem reasons that
# do not apply here, so bypass the check rather than change the pin.
%pip install -q --ignore-requires-python -e ".[dev]"

Cloning into 'panorama-onco'...
remote: Enumerating objects: 406, done.
remote: Counting objects: 100% (252/252), done.
remote: Compressing objects: 100% (148/148), done.
remote: Total 406 (delta 114), reused 215 (delta 86), pack-reused 154 (from 1)
Receiving objects: 100% (406/406), 142.49 KiB | 5.94 MiB/s, done.
Resolving deltas: 100% (177/177), done.
/content/panorama-onco
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983

In [4]:
import sys, os
print("cwd:", os.getcwd())
print("src exists:", os.path.exists('/content/panorama-onco/src/panorama'))
!pip show panorama-onco 2>/dev/null | head -3

# The editable install writes a path file that the ALREADY-RUNNING kernel has
# not read. Point at src/ directly -- simpler than restarting the runtime.
if '/content/panorama-onco/src' not in sys.path:
    sys.path.insert(0, '/content/panorama-onco/src')

import panorama
from panorama.vision.encoder import MultiStreamViT
from panorama.data.dicom import read_series
print("panorama imports OK")

cwd: /content/panorama-onco
src exists: True
Name: panorama-onco
Version: 0.0.1
Summary: 
panorama imports OK


In [6]:
# Cell 3 -- regenerate the data (ADR-0006: the recipe is versioned, not the data)
!python scripts/build_synthetic_cohort.py --patients 200 --max-studies 4
!python scripts/build_report_corpus.py

2026-08-25 11:24:20 | INFO     | panorama.data.synthetic | wrote 702 studies for 200 patients to data/synthetic/raw
2026-08-25 11:24:20 | INFO     | panorama.data.synthetic | lesion ground truth: data/synthetic/manifests/lesions.csv (1737 lesions)
2026-08-25 11:24:20 | INFO     | panorama.data.manifest | scanned 702 studies (0 files skipped)
2026-08-25 11:24:20 | INFO     | __main__ | manifest: data/synthetic/manifests/cohort.csv (702 studies, 200 patients)
2026-08-25 11:24:20 | INFO     | panorama.clinical.corpus | built 702 reports (0 studies had no lesion data)
2026-08-25 11:24:21 | INFO     | panorama.clinical.corpus | corpus written: data/synthetic/manifests/reports.jsonl (702 reports)


In [8]:
# Cell 4 -- real data, if you want it (~1.3 GB for 10 patients, more for 38)
!python scripts/download_tcia.py --collection QIN-BREAST --patients 38
!python scripts/convert_dicom.py
!python -c "from panorama.data.manifest import scan_directory, write_manifest; from panorama.core.logging import configure_logging; configure_logging('INFO'); write_manifest(scan_directory('data/tcia/qin-breast-nifti'), 'data/tcia/manifests/qin-breast.csv')"

2026-08-25 11:26:49 | INFO     | __main__ | QIN-BREAST: 68 patients
2026-08-25 11:27:58 | WARNING  | __main__ | request failed (ReadTimeout); retrying in 1.2s [1/5]
2026-08-25 11:30:34 | WARNING  | __main__ | request failed (ReadTimeout); retrying in 1.7s [1/5]
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py", line 787, in urlopen
    response = self._make_request(
        conn,
    ...<10 lines>...
        **response_kw,
    )
  File "/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
  File "/usr/local/lib/python3.13/dist-packages/urllib3/connection.py", line 565, in getresponse
    httplib_response = super().getresponse()
  File "/usr/lib/python3.13/http/client.py", line 1478, in getresponse
    response.begin()
    ~~~~~~~~~~~~~~^^
  File "/usr/lib/python3.13/http/client.py", line 343, in begin
    version, status, reason = self._read_status()
 

In [7]:
# Cell 5 -- the experiment ADR-0009 points to: does capacity fix under-learning?
!python -m panorama.train.pretrain configs/pretrain_qin.yaml \
  model.embed_dim=768 model.depth=12 model.num_heads=12 \
  model.decoder_dim=512 model.decoder_depth=4 \
  data.batch_size=8 data.num_workers=2 \
  model.max_steps=30000 trainer.max_steps=30000 model.warmup_steps=1000 \
  trainer.accelerator=gpu trainer.precision=16-mixed

2026-08-25 11:24:35 | INFO     | __main__ | resolved config:
seed: 1337
output_dir: outputs/qin
data:
  manifest_path: data/tcia/manifests/qin-breast.csv
  data_root: data/tcia/qin-breast-nifti
  crop_size:
  - 32
  - 32
  - 32
  target_spacing:
  - 2.0
  - 2.0
  - 2.0
  batch_size: 8
  num_workers: 2
  patches_per_study: 4
  fg_threshold: 0.5
  val_fraction: 0.2
  test_fraction: 0.1
model:
  volume_shape:
  - 32
  - 32
  - 32
  patch_size: 8
  embed_dim: 768
  depth: 12
  num_heads: 12
  fusion_every: 2
  share_stream_weights: true
  mask_ratio: 0.75
  decoder_dim: 512
  decoder_depth: 4
  decoder_heads: 4
  norm_pix_loss: true
  base_lr: 0.001
  weight_decay: 0.05
  warmup_steps: 1000
  max_steps: 30000
trainer:
  max_steps: 30000
  accelerator: gpu
  devices: 1
  precision: 16-mixed
  accumulate_grad_batches: 1
  gradient_clip_val: 1.0
  check_val_every_n_epoch: null
  val_check_interval: 100
  log_every_n_steps: 20

2026-08-25 11:24:35 | INFO     | __main__ | run directory: outputs

In [ ]:
# Cell 6 -- save results back, since the VM is wiped on disconnect
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/panorama-runs
!cp -r outputs/qin /content/drive/MyDrive/panorama-runs/